# B2S 03 - AndinaLog WMS Orders

Conversión auditada de órdenes WMS desde Bronze a Silver y cuarentena. Las fuentes de entrada se leen sin modificarse; Productos Silver y Flota Silver se usan exclusivamente para validación referencial.

In [1]:
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 120)


def find_root():
    candidates = []
    if os.getenv("ANDINALOG_ROOT"):
        candidates.append(Path(os.environ["ANDINALOG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "datos" / "bronze").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró el directorio datos/bronze")


ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    "entidad": "orden_wms",
    "granularidad": "una fila por order_id normalizado",
    "clave": "order_id",
    "rutas": {
        "bronze": "datos/bronze/andinalog_wms_orders.csv",
        "productos_silver": "datos/silver/andinalog_productos_silver.csv",
        "flota_silver": "datos/silver/andinalog_flota_silver.csv",
        "notebook": "notebooks/bronze_silver/03_wms_orders/B2S_03_AndinaLog_WMS_Orders.ipynb",
        "silver": "datos/silver/andinalog_wms_orders_silver.csv",
        "quarantine": "datos/quarantine/andinalog_wms_orders_quarantine.csv",
        "informe": "informes/bronze_silver/Informe_B2S_03_WMS_Orders.md",
    },
    "lectura": {"encoding": "utf-8", "dtype": "str", "keep_default_na": False},
    "columnas": {
        "identificadores": ["order_id", "cliente_id", "producto_id", "camion_id", "chofer_id"],
        "texto": ["centro_distribucion"],
        "fecha": "fecha_despacho",
        "numericas": [
            "cantidad_solicitada", "cantidad_entregada",
            "tiempo_entrega_prometido_hrs", "tiempo_entrega_real_hrs",
            "otif_on_time", "otif_in_full", "otif",
        ],
        "binarias": ["otif_on_time", "otif_in_full", "otif"],
        "obligatorias": [
            "order_id", "cliente_id", "producto_id", "fecha_despacho",
            "centro_distribucion", "camion_id", "chofer_id",
            "cantidad_solicitada", "cantidad_entregada",
            "tiempo_entrega_prometido_hrs", "tiempo_entrega_real_hrs",
            "otif_on_time", "otif_in_full", "otif",
        ],
        "permitir_extra": False,
    },
    "formatos_identificador": {
        "order_id": r"^ORD-2026-\d{5}$",
        "cliente_id": r"^CLI-\d{3}$",
        "producto_id": r"^PROD-\d{3}$",
        "camion_id": r"^CAM-\d{2}$",
        "chofer_id": r"^CHO-\d{3}$",
    },
    "formatos_fecha": {
        "iso": {"regex": r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$", "format": "%Y-%m-%d %H:%M:%S"},
        "dmy": {"regex": r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$", "format": "%d/%m/%Y %H:%M"},
    },
    "zonas_horarias": {"sin_zona": "America/La_Paz", "silver": "UTC"},
    "centros_validos": ["Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"],
    "centinelas": [-999],
    "numeros_escritos": {"cantidad_solicitada": {"cincuenta": 50}},
    "reglas_numericas": {
        "cantidad_solicitada_positiva": True,
        "cantidad_entregada_no_negativa": True,
        "tiempo_prometido_positivo": True,
        "tiempo_real_no_negativo": True,
    },
    "imputaciones": {"habilitadas": False, "metodo": "", "motivo": "no existe regla inequívoca aprobada"},
    "politica_duplicados": {"metodo": "cuarentena_de_todas_las_ocurrencias", "motivo": "no elegir arbitrariamente una orden duplicada"},
    "integridad_referencial": {
        "producto_id": {"fuente": "productos_silver", "tratamiento_sin_correspondencia": "cuarentena"},
        "camion_id": {"fuente": "flota_silver", "tratamiento_sin_correspondencia": "cuarentena"},
    },
    "semilla": None,
    "contradicciones_plan": [
        "la fuente tiene 7550 filas y 14 columnas, no 7551 y 15",
        "order_id usa cinco dígitos secuenciales, no tres",
        "no existe viaje_id; no se construye una relación de viaje sintética",
    ],
}
PATHS = {name: ROOT / relative for name, relative in CONFIG["rutas"].items()}
print("Raíz:", ROOT)
print("Fecha de ejecución UTC:", EXECUTED_AT_UTC)
print("Contradicciones documentadas:", CONFIG["contradicciones_plan"])


Raíz: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Fecha de ejecución UTC: 2026-09-24T18:02:51.407405+00:00
Contradicciones documentadas: ['la fuente tiene 7550 filas y 14 columnas, no 7551 y 15', 'order_id usa cinco dígitos secuenciales, no tres', 'no existe viaje_id; no se construye una relación de viaje sintética']


In [2]:
bronze = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])
productos_silver = pd.read_csv(PATHS["productos_silver"], **CONFIG["lectura"])
flota_silver = pd.read_csv(PATHS["flota_silver"], **CONFIG["lectura"])
normalized_order = bronze["order_id"].str.strip().str.upper()
numeric_profile = {
    column: pd.to_numeric(bronze[column], errors="coerce")
    for column in CONFIG["columnas"]["numericas"]
}
date_iso = bronze["fecha_despacho"].str.match(CONFIG["formatos_fecha"]["iso"]["regex"])
date_dmy = bronze["fecha_despacho"].str.match(CONFIG["formatos_fecha"]["dmy"]["regex"])
perfil = {
    "filas": len(bronze),
    "columnas": len(bronze.columns),
    "nombres_columnas": bronze.columns.tolist(),
    "tipos_recibidos": bronze.dtypes.astype(str).to_dict(),
    "vacios": bronze.eq("").sum().to_dict(),
    "duplicados_exactos_filas": int(bronze.duplicated(keep=False).sum()),
    "duplicados_clave_filas": int(normalized_order.duplicated(keep=False).sum()),
    "duplicados_clave_cantidad": int(normalized_order[normalized_order.duplicated(keep=False)].nunique()),
    "espacios_identificadores": {column: int(bronze[column].ne(bronze[column].str.strip()).sum()) for column in CONFIG["columnas"]["identificadores"]},
    "formatos_fecha": {"iso": int(date_iso.sum()), "dmy": int(date_dmy.sum()), "no_reconocido": int((~(date_iso | date_dmy)).sum())},
    "numericos_no_convertibles": {column: int((series.isna() & bronze[column].ne("")).sum()) for column, series in numeric_profile.items()},
    "centros": bronze["centro_distribucion"].value_counts(dropna=False).to_dict(),
}
print("Perfil Bronze")
for key, value in perfil.items():
    print(f"{key}: {value}")
print("Productos Silver disponibles:", productos_silver["producto_id"].nunique())
print("Camiones Silver disponibles:", flota_silver["camion_id"].nunique())


Perfil Bronze
filas: 7550
columnas: 14
nombres_columnas: ['order_id', 'cliente_id', 'producto_id', 'fecha_despacho', 'centro_distribucion', 'camion_id', 'chofer_id', 'cantidad_solicitada', 'cantidad_entregada', 'tiempo_entrega_prometido_hrs', 'tiempo_entrega_real_hrs', 'otif_on_time', 'otif_in_full', 'otif']
tipos_recibidos: {'order_id': 'str', 'cliente_id': 'str', 'producto_id': 'str', 'fecha_despacho': 'str', 'centro_distribucion': 'str', 'camion_id': 'str', 'chofer_id': 'str', 'cantidad_solicitada': 'str', 'cantidad_entregada': 'str', 'tiempo_entrega_prometido_hrs': 'str', 'tiempo_entrega_real_hrs': 'str', 'otif_on_time': 'str', 'otif_in_full': 'str', 'otif': 'str'}
vacios: {'order_id': 0, 'cliente_id': 0, 'producto_id': 0, 'fecha_despacho': 0, 'centro_distribucion': 0, 'camion_id': 0, 'chofer_id': 0, 'cantidad_solicitada': 0, 'cantidad_entregada': 80, 'tiempo_entrega_prometido_hrs': 0, 'tiempo_entrega_real_hrs': 0, 'otif_on_time': 0, 'otif_in_full': 0, 'otif': 0}
duplicados_exactos

In [3]:
def append_reason(df, mask, column, reason):
    df.loc[mask, column] = df.loc[mask, column].map(
        lambda current: reason if not current else f"{current} | {reason}"
    )
    return df


def validar_contrato_entrada(df):
    df = df.copy()
    expected = set(CONFIG["columnas"]["obligatorias"])
    received = set(df.columns)
    missing = sorted(expected - received)
    extra = sorted(received - expected)
    if missing or (extra and not CONFIG["columnas"]["permitir_extra"]):
        raise ValueError(f"Contrato inválido; faltantes={missing}, extra={extra}")
    return df


def estructurar(df):
    df = df.copy()
    df["_fila_bronze"] = range(2, len(df) + 2)
    for column in ["errores_bloqueantes", "motivos_transformacion", "motivos_imputacion"]:
        df[column] = ""
    source_columns = CONFIG["columnas"]["obligatorias"]
    for column in source_columns:
        df[f"{column}_original"] = df[column]
    return df


def normalizar_texto(df):
    df = df.copy()
    for column in CONFIG["columnas"]["identificadores"]:
        treated = f"{column}_tratado"
        flag = f"{column}_transformado"
        df[treated] = df[column].str.strip().str.upper()
        df[flag] = df[column].ne(df[treated])
        append_reason(df, df[flag], "motivos_transformacion", f"normalizacion_identificador:{column}")
    df["centro_distribucion_tratado"] = df["centro_distribucion"].str.strip()
    df["centro_distribucion_transformado"] = df["centro_distribucion"].ne(df["centro_distribucion_tratado"])
    append_reason(df, df["centro_distribucion_transformado"], "motivos_transformacion", "normalizacion_texto:centro_distribucion")
    return df


def normalizar_numeros_escritos(df):
    df = df.copy()
    for column, mapping in CONFIG["numeros_escritos"].items():
        treated_text = f"{column}_texto_tratado"
        flag = f"{column}_palabra_normalizada"
        normalized_text = df[column].str.strip().str.casefold()
        df[treated_text] = df[column]
        df[flag] = normalized_text.isin(mapping)
        df.loc[df[flag], treated_text] = normalized_text[df[flag]].map(mapping).astype(str)
        append_reason(df, df[flag], "motivos_transformacion", f"numero_escrito_normalizado:{column}")
    return df


def convertir_numericos(df):
    df = df.copy()
    for column in CONFIG["columnas"]["numericas"]:
        source = f"{column}_texto_tratado" if f"{column}_texto_tratado" in df.columns else column
        treated = f"{column}_tratado"
        invalid_flag = f"{column}_conversion_invalida"
        sentinel_flag = f"{column}_centinela_detectado"
        df[treated] = pd.to_numeric(df[source], errors="coerce")
        df[invalid_flag] = df[treated].isna() & df[column].ne("")
        append_reason(df, df[invalid_flag], "errores_bloqueantes", f"conversion_invalida:{column}")
        df[sentinel_flag] = df[treated].isin(CONFIG["centinelas"])
        df.loc[df[sentinel_flag], treated] = pd.NA
        append_reason(df, df[sentinel_flag], "errores_bloqueantes", f"centinela:{column}")
    return df


def convertir_fecha(df):
    df = df.copy()
    column = CONFIG["columnas"]["fecha"]
    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    recognized = pd.Series(False, index=df.index)
    df["fecha_despacho_formato"] = ""
    for name, rule in CONFIG["formatos_fecha"].items():
        mask = df[column].str.match(rule["regex"])
        parsed.loc[mask] = pd.to_datetime(df.loc[mask, column], format=rule["format"], errors="coerce")
        recognized = recognized | mask
        df.loc[mask, "fecha_despacho_formato"] = name
    df["fecha_despacho_formato_reconocido"] = recognized
    localized = parsed.dt.tz_localize(CONFIG["zonas_horarias"]["sin_zona"], ambiguous="NaT", nonexistent="NaT")
    df["fecha_despacho_tratado"] = localized.dt.tz_convert(CONFIG["zonas_horarias"]["silver"])
    df["fecha_despacho_conversion_invalida"] = df["fecha_despacho_tratado"].isna()
    append_reason(df, df["fecha_despacho_conversion_invalida"], "errores_bloqueantes", "fecha_despacho_invalida")
    df["fecha_despacho_transformado"] = ~df["fecha_despacho_conversion_invalida"]
    append_reason(df, df["fecha_despacho_transformado"], "motivos_transformacion", "fecha_local_bolivia_convertida_utc")
    return df


def validar_claves_y_catalogos(df):
    df = df.copy()
    for column, pattern in CONFIG["formatos_identificador"].items():
        invalid = ~df[f"{column}_tratado"].fillna("").str.match(pattern)
        append_reason(df, invalid, "errores_bloqueantes", f"formato_invalido:{column}")
    key = f"{CONFIG['clave']}_tratado"
    duplicate = df[key].duplicated(keep=False) & df[key].notna()
    append_reason(df, duplicate, "errores_bloqueantes", "order_id_duplicado")
    invalid_center = ~df["centro_distribucion_tratado"].isin(CONFIG["centros_validos"])
    append_reason(df, invalid_center, "errores_bloqueantes", "centro_distribucion_invalido")
    return df


def validar_numericos_y_otif(df):
    df = df.copy()
    requested = df["cantidad_solicitada_tratado"]
    delivered = df["cantidad_entregada_tratado"]
    promised = df["tiempo_entrega_prometido_hrs_tratado"]
    real = df["tiempo_entrega_real_hrs_tratado"]
    on_time = df["otif_on_time_tratado"]
    in_full = df["otif_in_full_tratado"]
    otif = df["otif_tratado"]
    append_reason(df, requested.notna() & requested.le(0), "errores_bloqueantes", "cantidad_solicitada_no_positiva")
    append_reason(df, delivered.notna() & delivered.lt(0), "errores_bloqueantes", "cantidad_entregada_negativa")
    append_reason(df, promised.notna() & promised.le(0), "errores_bloqueantes", "tiempo_prometido_no_positivo")
    append_reason(df, real.notna() & real.lt(0), "errores_bloqueantes", "tiempo_real_negativo")
    append_reason(df, requested.notna() & delivered.notna() & delivered.gt(requested), "errores_bloqueantes", "cantidad_entregada_mayor_solicitada")
    for column in CONFIG["columnas"]["binarias"]:
        invalid = df[f"{column}_tratado"].notna() & ~df[f"{column}_tratado"].isin([0, 1])
        append_reason(df, invalid, "errores_bloqueantes", f"indicador_no_binario:{column}")
    comparable_time = promised.notna() & real.notna()
    comparable_full = requested.notna() & delivered.notna()
    comparable_otif = on_time.notna() & in_full.notna() & otif.notna()
    append_reason(df, comparable_time & on_time.ne(real.le(promised).astype(int)), "errores_bloqueantes", "otif_on_time_incoherente")
    append_reason(df, comparable_full & in_full.ne(delivered.ge(requested).astype(int)), "errores_bloqueantes", "otif_in_full_incoherente")
    append_reason(df, comparable_otif & otif.ne((on_time * in_full).astype(int)), "errores_bloqueantes", "otif_incoherente")
    return df


def validar_integridad_referencial(df):
    df = df.copy()
    product_keys = set(productos_silver["producto_id"].str.strip().str.upper())
    fleet_keys = set(flota_silver["camion_id"].str.strip().str.upper())
    df["producto_id_corresponde_silver"] = df["producto_id_tratado"].isin(product_keys)
    df["camion_id_corresponde_silver"] = df["camion_id_tratado"].isin(fleet_keys)
    append_reason(df, ~df["producto_id_corresponde_silver"], "errores_bloqueantes", "producto_id_sin_correspondencia_silver")
    append_reason(df, ~df["camion_id_corresponde_silver"], "errores_bloqueantes", "camion_id_sin_correspondencia_silver")
    return df


def imputar(df):
    df = df.copy()
    df["fue_imputada"] = False
    df["imputacion_metodo"] = ""
    df["imputacion_motivo"] = ""
    return df


def validar_requeridos(df):
    df = df.copy()
    for column in CONFIG["columnas"]["obligatorias"]:
        treated = f"{column}_tratado"
        if column in CONFIG["columnas"]["identificadores"] or column in CONFIG["columnas"]["texto"]:
            missing = df[treated].eq("")
        else:
            missing = df[f"{column}_original"].str.strip().eq("")
        append_reason(df, missing, "errores_bloqueantes", f"requerido_ausente:{column}")
    return df


def asignar_calidad(df):
    df = df.copy()
    transform_flags = [f"{column}_transformado" for column in CONFIG["columnas"]["identificadores"]]
    transform_flags += ["centro_distribucion_transformado", "fecha_despacho_transformado", "cantidad_solicitada_palabra_normalizada"]
    df["fue_transformada"] = df[transform_flags].any(axis=1)
    df["conteo_transformaciones"] = df[transform_flags].sum(axis=1).astype(int)
    df["conteo_imputaciones"] = df["fue_imputada"].astype(int)
    df["calidad_motivo"] = df["errores_bloqueantes"].mask(df["errores_bloqueantes"].eq(""), "sin_incidencias")
    df["calidad_estado"] = "valida"
    df.loc[df["fue_transformada"], "calidad_estado"] = "valida_con_transformacion"
    df.loc[df["fue_imputada"], "calidad_estado"] = "valida_con_imputacion"
    df.loc[df["fue_transformada"] & df["fue_imputada"], "calidad_estado"] = "valida_con_transformacion_e_imputacion"
    df.loc[df["errores_bloqueantes"].ne(""), "calidad_estado"] = "cuarentena"
    return df


work = (
    bronze.pipe(validar_contrato_entrada)
    .pipe(estructurar)
    .pipe(normalizar_texto)
    .pipe(normalizar_numeros_escritos)
    .pipe(convertir_numericos)
    .pipe(convertir_fecha)
    .pipe(validar_claves_y_catalogos)
    .pipe(validar_numericos_y_otif)
    .pipe(validar_integridad_referencial)
    .pipe(imputar)
    .pipe(validar_requeridos)
    .pipe(asignar_calidad)
)
silver = work.loc[work["calidad_estado"].ne("cuarentena")].copy()
quarantine = work.loc[work["calidad_estado"].eq("cuarentena")].copy()
output_columns = CONFIG["columnas"]["identificadores"] + CONFIG["columnas"]["texto"] + CONFIG["columnas"]["numericas"]
for column in output_columns:
    silver[column] = silver[f"{column}_tratado"]
    quarantine[column] = quarantine[f"{column}_tratado"]
silver["fecha_despacho"] = silver["fecha_despacho_tratado"]
quarantine["fecha_despacho"] = quarantine["fecha_despacho_tratado"]
silver.to_csv(PATHS["silver"], index=False, encoding="utf-8")
quarantine.to_csv(PATHS["quarantine"], index=False, encoding="utf-8")
print({"bronze": len(bronze), "silver": len(silver), "quarantine": len(quarantine)})
print("Estados Silver:", silver["calidad_estado"].value_counts().to_dict())


{'bronze': 7550, 'silver': 6261, 'quarantine': 1289}
Estados Silver: {'valida_con_transformacion': 6261}


In [4]:
silver_file = pd.read_csv(PATHS["silver"])
quarantine_file = pd.read_csv(PATHS["quarantine"])
assert len(bronze) == len(silver_file) + len(quarantine_file)
assert set(silver_file["_fila_bronze"]).isdisjoint(set(quarantine_file["_fila_bronze"]))
assert set(silver_file["_fila_bronze"]) | set(quarantine_file["_fila_bronze"]) == set(work["_fila_bronze"])
assert silver_file["order_id"].is_unique
assert silver_file["errores_bloqueantes"].fillna("").eq("").all()
assert silver_file["producto_id_corresponde_silver"].all()
assert silver_file["camion_id_corresponde_silver"].all()
assert silver_file["fecha_despacho"].str.endswith("+00:00").all()
assert not silver_file["fue_imputada"].any()

def reason_counts(series):
    counts = {}
    for value in series.fillna(""):
        for reason in [item.strip() for item in value.split("|") if item.strip()]:
            counts[reason] = counts.get(reason, 0) + 1
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))

quality = silver_file["calidad_estado"].value_counts().to_dict()
reasons = reason_counts(quarantine_file["errores_bloqueantes"])
transformed_words = int(work["cantidad_solicitada_palabra_normalizada"].sum())
unmatched_products = int((~work["producto_id_corresponde_silver"]).sum())
unmatched_fleet = int((~work["camion_id_corresponde_silver"]).sum())
invalid_dates = int(work["fecha_despacho_conversion_invalida"].sum())
report = [
    "# Informe B2S 03 - AndinaLog WMS Orders", "",
    "## Objetivo, entidad y granularidad",
    "Conversión auditada de órdenes WMS desde Bronze a Silver y cuarentena.",
    f"- Entidad: {CONFIG['entidad']}.",
    f"- Granularidad: {CONFIG['granularidad']}.",
    "- No existe `viaje_id` en la fuente y no se construye una relación sintética.", "",
    "## Contrato y perfil Bronze",
    f"- Columnas obligatorias: {CONFIG['columnas']['obligatorias']}.",
    f"- Filas: {perfil['filas']}; columnas: {perfil['columnas']}.",
    f"- Tipos recibidos: {perfil['tipos_recibidos']}.",
    f"- Valores vacíos: {perfil['vacios']}.",
    f"- Duplicados exactos: {perfil['duplicados_exactos_filas']} filas; duplicados de clave: {perfil['duplicados_clave_filas']} filas en {perfil['duplicados_clave_cantidad']} claves.",
    f"- Espacios en identificadores: {perfil['espacios_identificadores']}.",
    f"- Formatos de fecha: {perfil['formatos_fecha']}.",
    f"- Textos no convertibles antes del tratamiento: {perfil['numericos_no_convertibles']}.",
    f"- Centros observados: {perfil['centros']}.", "",
    "## Transformaciones, fechas e imputación",
    "- Identificadores se recortan y normalizan a mayúsculas con auditoría.",
    f"- `cincuenta` se normaliza inequívocamente a 50 en {transformed_words} filas y conserva el original y la bandera.",
    "- Fechas ISO y DD/MM/YYYY se interpretan en America/La_Paz y se convierten a UTC; las fechas imposibles permanecen en cuarentena.",
    f"- Fechas inválidas: {invalid_dates}.",
    "- No se imputan identificadores, cantidades, tiempos, fechas ni indicadores; no hay información suficiente para hacerlo inequívocamente.", "",
    "## OTIF y coherencia",
    "- `otif_on_time` debe concordar con tiempo real <= prometido; `otif_in_full` con cantidad entregada >= solicitada; `otif` con el producto de ambos indicadores.",
    "- Cantidades y tiempos inválidos o incoherentes permanecen en cuarentena.", "",
    "## Integridad referencial",
    f"- Filas sin producto en Productos Silver: {unmatched_products}.",
    f"- Filas sin camión en Flota Silver: {unmatched_fleet}.",
    "- No se inventan correspondencias; cada ausencia se registra y es bloqueante.", "",
    "## Contradicciones respecto del plan",
    *[f"- {item}." for item in CONFIG["contradicciones_plan"]], "",
    "## Resultado y conciliación",
    f"- Silver: {len(silver_file)} filas; estados: {quality}.",
    f"- Cuarentena: {len(quarantine_file)} filas; motivos por regla: {reasons}.",
    f"- Conciliación persistida: Bronze {len(bronze)} = Silver {len(silver_file)} + cuarentena {len(quarantine_file)}.",
    "- Silver tiene clave única, referencias válidas y cero errores bloqueantes.",
    "- Silver y cuarentena son disjuntos y su unión cubre todas las filas Bronze.", "",
    "## Archivos generados",
    f"- `{CONFIG['rutas']['notebook']}`",
    f"- `{CONFIG['rutas']['silver']}`",
    f"- `{CONFIG['rutas']['quarantine']}`",
    f"- `{CONFIG['rutas']['informe']}`", "",
    "## Reproducibilidad",
    f"- Fecha de ejecución UTC: {EXECUTED_AT_UTC}.",
    f"- Python: {platform.python_version()}.",
    f"- pandas: {pd.__version__}.",
    f"- Rutas de entrada: `{CONFIG['rutas']['bronze']}`, `{CONFIG['rutas']['productos_silver']}`, `{CONFIG['rutas']['flota_silver']}`.",
    f"- Rutas de salida: `{CONFIG['rutas']['silver']}`, `{CONFIG['rutas']['quarantine']}`, `{CONFIG['rutas']['informe']}`.",
    f"- Zona inicial: {CONFIG['zonas_horarias']['sin_zona']}; zona Silver: {CONFIG['zonas_horarias']['silver']}.",
    f"- Conteos: Bronze {len(bronze)}, Silver {len(silver_file)}, cuarentena {len(quarantine_file)}.",
    f"- Semilla: {CONFIG['semilla']} (no aplica; el pipeline es determinístico).",
    "- Las entradas no se modifican; los controles finales se ejecutan sobre los CSV persistidos.",
]
PATHS["informe"].write_text("\n".join(report) + "\n", encoding="utf-8")
print("Controles persistidos: OK")
print(pd.DataFrame({"bronze": [len(bronze)], "silver": [len(silver_file)], "quarantine": [len(quarantine_file)]}))
print("Informe:", PATHS["informe"])


Controles persistidos: OK
   bronze  silver  quarantine
0    7550    6261        1289
Informe: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_03_WMS_Orders.md
